In [1]:
# (C) Martin Reißel

import numpy as np
import sympy as sy

from IPython.display import display, Math, Latex

platex = lambda A: sy.latex(A, mat_str='pmatrix', mat_delim='')

# Aufgabenstellung

Steepest-Descent mit $M = I$ für

In [2]:
A = sy.Matrix([[2,1],[1,2]])
x = sy.Matrix([2, 1])
b = A*x
x0 = sy.Matrix([0, 2])

Math(r'{} x = {}'.format(platex(A), platex(b)))

<IPython.core.display.Math object>

und

In [3]:
Math(r'x_0 = {}'.format(platex(x0)))

<IPython.core.display.Math object>

## Lösung

Steepest Descend für LGS: starte mit $x_0$, $r_0 = b - Ax_0$ und wiederhole
\begin{align*}
p_k &= M^{-1} r_k\\
s_k &= Ap_k\\
\alpha_k &= \frac{<p_k, r_k>}{<p_k, s_k>}\\
x_{k+1} &= x_k + \alpha_k p_k\\
r_{k+1} &= r_k - \alpha_k s_k
\end{align*}
bzw. mit $M=I$ ($p_k = r_k$)
\begin{align*}
s_k &= Ar_k\\
\alpha_k &= \frac{<r_k, r_k>}{<r_k, s_k>}\\
x_{k+1} &= x_k + \alpha_k r_k\\
r_{k+1} &= r_k - \alpha_k s_k
\end{align*}

In [4]:
def sd(xk, nit = 4):
    rk = b - A*xk
    for k in range(nit):
        sk = A*rk
        ak = rk.dot(rk) / (rk.dot(sk))
        display(Math(r'x_{0} = {1},\quad r_{0}= {2},\quad s_{0} = {3},\quad \alpha_{0}= {4}'.format(k, platex(xk), platex(rk), platex(sk), platex(ak))))
        xk = xk + ak*rk
        rk = rk - ak*sk
        
    return xk
        
xk = sd(x0)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
xk.evalf()

Matrix([
[ 1.875],
[1.0625]])

## "Nichtlineares SD" mit exakter Liniensuche

In [6]:
x1, x2 = sy.symbols("x1,x2", real = True)
x = sy.Matrix([x1, x2])
x12 = (x1, x2)

In [7]:
f = sy.Lambda(x12, (x.T * A * x / 2 - b.T * x)[0].expand())
f

Lambda((x1, x2), x1**2 + x1*x2 - 5*x1 + x2**2 - 4*x2)

Bei exakter Liniensuche wird $\alpha$ bestimmt durch
$$
\varphi(\alpha) = \min_{s\geq0} \varphi(s),
\quad
\varphi(s) = f(x - s f'(x)).
$$
Mit

In [8]:
f1 = sy.Lambda(x12, sy.Matrix([f(*x12)]).jacobian(x12))
Math(r"f'(x) = " + platex(f1(*x12).T) + r",\quad x = " + platex(x0))

<IPython.core.display.Math object>

folgt

In [9]:
xk = x0

#### Hier startet die Iteration

In [10]:
s = sy.symbols('s')
xs = xk - s*f1(*xk).T

Math(r"x - sf'(x) = " + platex(xs))

<IPython.core.display.Math object>

und deshalb

In [11]:
phi = sy.Lambda(s, f(*xs).simplify())

Math(r"\varphi(s) = f(x - sf'(x)) = " + platex(phi(s)))

<IPython.core.display.Math object>

Um $\varphi$ über $[0,\infty)$ zu minimieren betrachten wir alle lokalen
Extrema sowie $\varphi(0)$ und $\lim_{s\to\infty}\varphi(s)$.
Für die lokalen Extrema erhalten wir mit

In [12]:
phi1 = sy.Lambda(s, phi(s).diff(s))
Math(r"\varphi'(s) = " + platex(phi1(s)) )

<IPython.core.display.Math object>

In [13]:
sex = sy.solve(phi1(s), s)
pex = list(map(phi, sex))
Math(r"\hat{s} \in" + sy.latex(sex) + r",\qquad \varphi(\hat{s}) \in" + sy.latex(pex))

<IPython.core.display.Math object>

und für $s=0$ bzw. $s\to\infty$

In [14]:
p0 = phi(0)
pinf = phi(s).limit(s, sy.oo)
Math(r"\varphi(0)=" + sy.latex(p0) + r",\qquad \lim_{s\to\infty}\varphi(s) =" + sy.latex(pinf))

<IPython.core.display.Math object>

In [15]:
ss = sex + [0]
pp = np.array(map(phi, ss))

smin = ss[pp.argmin()]
if pinf < phi(smin):
    smin = sy.oo

al = smin

also

In [16]:
Math(r'\alpha =' + sy.latex(al))

<IPython.core.display.Math object>

und somit

In [17]:
xn = xk - al * f1(*xk).T
Math(r"x_\text{neu} = x - \alpha f'(x) = " 
     + platex(xk) + "-" + sy.latex(al) + platex(f1(*x0).T)
     + "=" + platex(xn))

<IPython.core.display.Math object>

und

In [18]:
display(Math(r'f(x) = {} \qquad f(x_\text{{neu}}) = {}'.format(f(*xk), f(*xn))))

xk = xn

<IPython.core.display.Math object>